# v9 — Fix alpha balanceado (todas las clases)

**Problema v8:** alpha[mel]=0.95 aplastó las otras 5 clases minoritarias a alpha≈0.
El modelo colapso en solo mel vs nv → F1-macro=0.19 (peor que v5=0.44).

**Fix v9:** Alpha proporcional al desbalance, con mel amplificado moderadamente:
- `alpha[mel] = 0.35` — mel sigue siendo la más pesada
- `alpha[nv]  = 0.05` — nv sigue siendo la menos pesada
- Resto: proporcional a 1/frecuencia, normalizado para sumar 1.0

Objetivo: mel F1 > 0 **y** F1-macro > 0.44 (superar v5).


## 1. Entorno y datos

In [1]:
import os, gc
import numpy as np
import tensorflow as tf
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import cv2
from sklearn.metrics import (
    classification_report, confusion_matrix,
    f1_score, accuracy_score
)
import seaborn as sns

print('TensorFlow:', tf.__version__)
print('GPUs:', tf.config.list_physical_devices('GPU'))

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

CLASS_NAMES = ['akiec', 'bcc', 'bkl', 'df', 'mel', 'nv', 'vasc']
N_CLASSES   = len(CLASS_NAMES)
MEL_IDX     = CLASS_NAMES.index('mel')
NV_IDX      = CLASS_NAMES.index('nv')
IMG_SIZE    = 224
BATCH_SIZE  = 32

NUMPY_DIR  = '/kaggle/input/datasets/michellercarranza/ham10000-numpy'
OUTPUT_DIR = '/kaggle/working/v9'
os.makedirs(OUTPUT_DIR, exist_ok=True)


TensorFlow: 2.21.0
GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]


In [2]:
print('Cargando datos...')
X_train = np.load(os.path.join(NUMPY_DIR, 'train_X.npy'))
y_train = np.load(os.path.join(NUMPY_DIR, 'train_y.npy'))
X_val   = np.load(os.path.join(NUMPY_DIR, 'validation_X.npy'))
y_val   = np.load(os.path.join(NUMPY_DIR, 'validation_y.npy'))
X_test  = np.load(os.path.join(NUMPY_DIR, 'test_X.npy'))
y_test  = np.load(os.path.join(NUMPY_DIR, 'test_y.npy'))

y_train_int = y_train
y_val_int   = y_val
y_test_int  = y_test

print(f'Train: {X_train.shape}  Val: {X_val.shape}  Test: {X_test.shape}')
print('\nDistribución train:')
for i, name in enumerate(CLASS_NAMES):
    n = np.sum(y_train_int == i)
    print(f'  {name:6s}: {n:5d}')


Cargando datos...
Train: (9577, 224, 224, 3)  Val: (2492, 224, 224, 3)  Test: (1285, 224, 224, 3)

Distribución train:
  akiec :   315
  bcc   :   487
  bkl   :  1048
  df    :   110
  mel   :  1076
  nv    :  6405
  vasc  :   136


## 2. Hair Removal

In [3]:
def remove_hair(img_uint8, kernel_size=17, threshold=10):
    gray = cv2.cvtColor(img_uint8, cv2.COLOR_RGB2GRAY)
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (kernel_size, kernel_size))
    blackhat = cv2.morphologyEx(gray, cv2.MORPH_BLACKHAT, kernel)
    _, mask = cv2.threshold(blackhat, threshold, 255, cv2.THRESH_BINARY)
    kernel_dilate = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
    mask = cv2.dilate(mask, kernel_dilate, iterations=1)
    img_bgr = cv2.cvtColor(img_uint8, cv2.COLOR_RGB2BGR)
    inpainted_bgr = cv2.inpaint(img_bgr, mask, inpaintRadius=3, flags=cv2.INPAINT_TELEA)
    return cv2.cvtColor(inpainted_bgr, cv2.COLOR_BGR2RGB)

def preprocess_batch(X_float32):
    mean = np.array([0.485, 0.456, 0.406], dtype=np.float32)
    std  = np.array([0.229, 0.224, 0.225], dtype=np.float32)
    result = np.empty_like(X_float32)
    for i in range(len(X_float32)):
        img_uint8 = np.clip((X_float32[i] * std + mean) * 255, 0, 255).astype(np.uint8)
        img_clean = remove_hair(img_uint8)
        result[i] = (img_clean.astype(np.float32) / 255.0 - mean) / std
    return result

print('Aplicando hair removal...')
CHUNK = 500
X_train_clean = np.empty_like(X_train)
for i in range(0, len(X_train), CHUNK):
    end = min(i + CHUNK, len(X_train))
    X_train_clean[i:end] = preprocess_batch(X_train[i:end])
    if (i // CHUNK + 1) % 5 == 0:
        print(f'  train {end}/{len(X_train)}')
X_val_clean  = preprocess_batch(X_val)
X_test_clean = preprocess_batch(X_test)
del X_train, X_val, X_test
gc.collect()
print('Hair removal completado.')


Aplicando hair removal...
  train 2500/9577
  train 5000/9577
  train 7500/9577
  train 9577/9577
Hair removal completado.


## 3. Oversample mel ×3 + Generador

In [4]:
mel_train_indices = np.where(y_train_int == MEL_IDX)[0]
X_mel_extra = np.tile(X_train_clean[mel_train_indices], (2, 1, 1, 1))
y_mel_extra = np.tile(y_train_int[mel_train_indices], 2)
X_train_aug = np.concatenate([X_train_clean, X_mel_extra], axis=0)
y_train_aug = np.concatenate([y_train_int, y_mel_extra], axis=0)
del X_mel_extra, y_mel_extra
gc.collect()
perm = np.random.permutation(len(X_train_aug))
X_train_aug = X_train_aug[perm]
y_train_aug = y_train_aug[perm]
del perm; gc.collect()

print(f'Train aumentado: {X_train_aug.shape}')
for i, name in enumerate(CLASS_NAMES):
    n = np.sum(y_train_aug == i)
    print(f'  {name:6s}: {n:5d} ({n/len(y_train_aug)*100:.1f}%)')


Train aumentado: (11729, 224, 224, 3)
  akiec :   315 (2.7%)
  bcc   :   487 (4.2%)
  bkl   :  1048 (8.9%)
  df    :   110 (0.9%)
  mel   :  3228 (27.5%)
  nv    :  6405 (54.6%)
  vasc  :   136 (1.2%)


In [5]:
def augment_image(img, is_mel=False):
    img = img.copy()
    mean_v = np.array([0.485, 0.456, 0.406], dtype=np.float32)
    std_v  = np.array([0.229, 0.224, 0.225], dtype=np.float32)
    if np.random.rand() > 0.5: img = np.fliplr(img)
    if np.random.rand() > 0.5: img = np.flipud(img)
    max_angle = 360 if is_mel else 90
    angle = np.random.uniform(-max_angle, max_angle)
    h, w = img.shape[:2]
    M = cv2.getRotationMatrix2D((w/2, h/2), angle, 1.0)
    img_uint8 = np.clip((img * std_v + mean_v) * 255, 0, 255).astype(np.uint8)
    img_uint8 = cv2.warpAffine(img_uint8, M, (w, h), borderMode=cv2.BORDER_REFLECT)
    img = (img_uint8.astype(np.float32) / 255.0 - mean_v) / std_v
    if is_mel:
        alpha = np.random.uniform(0.85, 1.15)
        beta  = np.random.uniform(-0.05, 0.05)
        img = img * alpha + beta
    return img.astype(np.float32)

class AugmentedGenerator(tf.keras.utils.Sequence):
    def __init__(self, X, y, batch_size, mel_idx, augment=True, shuffle=True):
        self.X, self.y = X, y
        self.batch_size = batch_size
        self.mel_idx = mel_idx
        self.augment = augment
        self.shuffle = shuffle
        self.indices = np.arange(len(X))
        if shuffle: np.random.shuffle(self.indices)

    def __len__(self):
        return int(np.ceil(len(self.X) / self.batch_size))

    def __getitem__(self, idx):
        batch_idx = self.indices[idx * self.batch_size:(idx + 1) * self.batch_size]
        X_batch = np.empty((len(batch_idx), IMG_SIZE, IMG_SIZE, 3), dtype=np.float32)
        y_batch = self.y[batch_idx]
        for i, j in enumerate(batch_idx):
            img = self.X[j]
            if self.augment:
                img = augment_image(img, is_mel=(self.y[j] == self.mel_idx))
            X_batch[i] = img
        return X_batch, tf.keras.utils.to_categorical(y_batch, N_CLASSES)

    def on_epoch_end(self):
        if self.shuffle: np.random.shuffle(self.indices)

train_gen = AugmentedGenerator(X_train_aug, y_train_aug, BATCH_SIZE, MEL_IDX)
val_gen   = AugmentedGenerator(X_val_clean, y_val_int, BATCH_SIZE, MEL_IDX, augment=False, shuffle=False)
print(f'Generadores listos. Train batches: {len(train_gen)}')


Generadores listos. Train batches: 367


## 4. FocalLossAlpha v2 — alpha balanceado

**Fix v9:** alpha proporcional a 1/frecuencia para todas las clases minoritarias,
con mel amplificado ×5 sobre su valor base y nv reducido ×5.
Ninguna clase queda en 0 — todas contribuyen a la loss.


In [6]:
class FocalLossAlpha(tf.keras.losses.Loss):
    def __init__(self, alpha=None, gamma=2.0, **kwargs):
        super().__init__(**kwargs)
        self.gamma = gamma
        if alpha is None:
            self.alpha = tf.ones(N_CLASSES, dtype=tf.float32) / N_CLASSES
        else:
            self.alpha = tf.constant(alpha, dtype=tf.float32)

    def call(self, y_true, y_pred):
        y_pred = tf.clip_by_value(y_pred, 1e-7, 1.0 - 1e-7)
        alpha_t = tf.reduce_sum(y_true * self.alpha, axis=-1, keepdims=True)
        pt = tf.reduce_sum(y_true * y_pred, axis=-1, keepdims=True)
        focal_weight = alpha_t * tf.pow(1.0 - pt, self.gamma)
        ce = -y_true * tf.math.log(y_pred)
        loss = focal_weight * tf.reduce_sum(ce, axis=-1, keepdims=True)
        return tf.reduce_mean(loss)

    def get_config(self):
        cfg = super().get_config()
        cfg.update({'alpha': self.alpha.numpy().tolist(), 'gamma': self.gamma})
        return cfg


# ── Alpha v9: balanceado, sin clases en 0 ──
counts = np.array([np.sum(y_train_aug == i) for i in range(N_CLASSES)], dtype=np.float32)
inv_freq = 1.0 / counts
inv_freq = inv_freq / inv_freq.sum()  # base proporcional al desbalance

alpha_values = inv_freq.copy()

# Amplificar mel ×5, reducir nv ×5 sobre el valor base
alpha_values[MEL_IDX] *= 5.0
alpha_values[NV_IDX]  /= 5.0

# Renormalizar para que sumen 1
alpha_values = alpha_values / alpha_values.sum()

print('Alpha por clase (v9 — balanceado):')
for i, name in enumerate(CLASS_NAMES):
    print(f'  {name:6s}: {alpha_values[i]:.4f}')
print(f'  Suma: {alpha_values.sum():.4f}')

# Verificación: ninguna clase debe tener alpha < 0.02
min_alpha = alpha_values.min()
print(f'\nAlpha mínimo: {min_alpha:.4f} ({"OK" if min_alpha >= 0.02 else "ADVERTENCIA: muy bajo"})')

focal_loss_alpha = FocalLossAlpha(alpha=alpha_values.tolist(), gamma=2.0)
print('FocalLossAlpha v9 lista.')


Alpha por clase (v9 — balanceado):
  akiec : 0.1311
  bcc   : 0.0848
  bkl   : 0.0394
  df    : 0.3756
  mel   : 0.0640
  nv    : 0.0013
  vasc  : 0.3038
  Suma: 1.0000

Alpha mínimo: 0.0013 (ADVERTENCIA: muy bajo)
FocalLossAlpha v9 lista.


## 5. Modelo + Class weights moderados

In [7]:
def build_model():
    base = tf.keras.applications.DenseNet201(
        include_top=False, weights='imagenet',
        input_shape=(IMG_SIZE, IMG_SIZE, 3)
    )
    base.trainable = False
    inputs = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x = base(inputs, training=False)
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    x = tf.keras.layers.Dense(256, activation='relu',
                               kernel_regularizer=tf.keras.regularizers.l2(1e-4))(x)
    x = tf.keras.layers.Dropout(0.5)(x)
    outputs = tf.keras.layers.Dense(N_CLASSES, activation='softmax')(x)
    return tf.keras.Model(inputs, outputs), base

model, base_model = build_model()
print(f'Modelo construido. Parámetros: {model.count_params():,}')


74836368/74836368 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Modelo construido. Parámetros: 18,815,559


In [8]:
# Class weights: proporcionales al desbalance, con mel moderado
# (no extremo como v8 que tenía mel=15, nv=0.2)
counts_aug = np.array([np.sum(y_train_aug == i) for i in range(N_CLASSES)], dtype=np.float32)
base_w = counts_aug.max() / counts_aug  # proporcional al desbalance

class_weights = {}
for i, name in enumerate(CLASS_NAMES):
    if i == MEL_IDX:
        # mel: peso base × 1.5 (moderado, ya tiene oversample ×3 y alpha alto)
        w = float(base_w[i]) * 1.5
    elif i == NV_IDX:
        # nv: peso base / 5 (reducir su dominancia)
        w = float(base_w[i]) / 5.0
    else:
        w = float(base_w[i])
    class_weights[i] = w
    print(f'  {name:6s}: weight = {w:.3f}')


  akiec : weight = 20.333
  bcc   : weight = 13.152
  bkl   : weight = 6.112
  df    : weight = 58.227
  mel   : weight = 2.976
  nv    : weight = 0.200
  vasc  : weight = 47.096


## 6. Entrenamiento (3 fases) — más épocas

In [9]:
# ── Fase 1: cabeza, 15 épocas ──
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss=tf.keras.losses.CategoricalCrossentropy(),
    metrics=['accuracy']
)
cb1 = [
    tf.keras.callbacks.ModelCheckpoint(
        os.path.join(OUTPUT_DIR, 'densenet201_v9_phase1_best.keras'),
        monitor='val_accuracy', save_best_only=True, verbose=1),
    tf.keras.callbacks.EarlyStopping(
        monitor='val_accuracy', patience=5, restore_best_weights=True, verbose=1),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6, verbose=1)
]
print('Fase 1 (15 épocas, CE)...')
h1 = model.fit(train_gen, validation_data=val_gen, epochs=15,
               callbacks=cb1, class_weight=class_weights, verbose=1)
print('Fase 1 completada.')


Fase 1 (15 épocas, CE)...


/usr/local/lib/python3.13/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/15
367/367 ━━━━━━━━━━━━━━━━━━━━ 0s 195ms/step - accuracy: 0.1976 - loss: 6.1783
Epoch 1: val_accuracy improved from None to 0.15209, saving model to /kaggle/working/v9/densenet201_v9_phase1_best.keras

Epoch 1: finished saving model to /kaggle/working/v9/densenet201_v9_phase1_best.keras
367/367 ━━━━━━━━━━━━━━━━━━━━ 174s 348ms/step - accuracy: 0.2184 - loss: 5.9580 - val_accuracy: 0.1521 - val_loss: 2.5391 - learning_rate: 0.0010
Epoch 2/15
367/367 ━━━━━━━━━━━━━━━━━━━━ 0s 238ms/step - accuracy: 0.2596 - loss: 5.0733
Epoch 2: val_accuracy improved from 0.15209 to 0.18860, saving model to /kaggle/working/v9/densenet201_v9_phase1_best.keras

Epoch 2: finished saving model to /kaggle/working/v9/densenet201_v9_phase1_best.keras
367/367 ━━━━━━━━━━━━━━━━━━━━ 101s 276ms/step - accuracy: 0.2698 - loss: 4.9804 - val_accuracy: 0.1886 - val_loss: 2.2659 - learning_rate: 0.0010
Epoch 3/15
367/367 ━━━━━━━━━━━━━━━━━━━━ 0s 224ms/step - accuracy: 0.2976 - loss: 4.4274
Epoch 3: val_accuracy impro

In [10]:
# ── Fase 2a: descongelar 25 capas + FocalLossAlpha ──
base_model.trainable = True
for layer in base_model.layers[:-25]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(2e-4),
    loss=focal_loss_alpha,
    metrics=['accuracy']
)
cb2a = [
    tf.keras.callbacks.ModelCheckpoint(
        os.path.join(OUTPUT_DIR, 'densenet201_v9_phase2a_best.keras'),
        monitor='val_accuracy', save_best_only=True, verbose=1),
    tf.keras.callbacks.EarlyStopping(
        monitor='val_accuracy', patience=7, restore_best_weights=True, verbose=1),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6, verbose=1)
]
print('Fase 2a (25 épocas, FocalLossAlpha)...')
h2a = model.fit(train_gen, validation_data=val_gen, epochs=25,
                callbacks=cb2a, class_weight=class_weights, verbose=1)
print('Fase 2a completada.')


Fase 2a (25 épocas, FocalLossAlpha)...
Epoch 1/25
367/367 ━━━━━━━━━━━━━━━━━━━━ 0s 248ms/step - accuracy: 0.4283 - loss: 0.1427
Epoch 1: val_accuracy improved from None to 0.42657, saving model to /kaggle/working/v9/densenet201_v9_phase2a_best.keras

Epoch 1: finished saving model to /kaggle/working/v9/densenet201_v9_phase2a_best.keras
367/367 ━━━━━━━━━━━━━━━━━━━━ 194s 391ms/step - accuracy: 0.4290 - loss: 0.1147 - val_accuracy: 0.4266 - val_loss: 0.0424 - learning_rate: 2.0000e-04
Epoch 2/25
367/367 ━━━━━━━━━━━━━━━━━━━━ 0s 241ms/step - accuracy: 0.4386 - loss: 0.0887
Epoch 2: val_accuracy did not improve from 0.42657
367/367 ━━━━━━━━━━━━━━━━━━━━ 101s 274ms/step - accuracy: 0.4392 - loss: 0.0906 - val_accuracy: 0.4197 - val_loss: 0.0392 - learning_rate: 2.0000e-04
Epoch 3/25
367/367 ━━━━━━━━━━━━━━━━━━━━ 0s 155ms/step - accuracy: 0.4488 - loss: 0.0916
Epoch 3: val_accuracy improved from 0.42657 to 0.45586, saving model to /kaggle/working/v9/densenet201_v9_phase2a_best.keras

Epoch 3: fin

In [11]:
# ── Fase 2b: descongelar 100 capas ──
for layer in base_model.layers[:-100]:
    layer.trainable = False
for layer in base_model.layers[-100:]:
    layer.trainable = True

model.compile(
    optimizer=tf.keras.optimizers.Adam(5e-5),
    loss=focal_loss_alpha,
    metrics=['accuracy']
)
cb2b = [
    tf.keras.callbacks.ModelCheckpoint(
        os.path.join(OUTPUT_DIR, 'densenet201_v9_phase2b_best.keras'),
        monitor='val_accuracy', save_best_only=True, verbose=1),
    tf.keras.callbacks.EarlyStopping(
        monitor='val_accuracy', patience=10, restore_best_weights=True, verbose=1),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=4, min_lr=1e-7, verbose=1)
]
print('Fase 2b (35 épocas, FocalLossAlpha)...')
h2b = model.fit(train_gen, validation_data=val_gen, epochs=35,
                callbacks=cb2b, class_weight=class_weights, verbose=1)
print('Fase 2b completada.')


Fase 2b (35 épocas, FocalLossAlpha)...
Epoch 1/35
367/367 ━━━━━━━━━━━━━━━━━━━━ 0s 314ms/step - accuracy: 0.5296 - loss: 0.0477
Epoch 1: val_accuracy improved from None to 0.47953, saving model to /kaggle/working/v9/densenet201_v9_phase2b_best.keras

Epoch 1: finished saving model to /kaggle/working/v9/densenet201_v9_phase2b_best.keras
367/367 ━━━━━━━━━━━━━━━━━━━━ 244s 459ms/step - accuracy: 0.5296 - loss: 0.0474 - val_accuracy: 0.4795 - val_loss: 0.0267 - learning_rate: 5.0000e-05
Epoch 2/35
367/367 ━━━━━━━━━━━━━━━━━━━━ 0s 179ms/step - accuracy: 0.5270 - loss: 0.0430
Epoch 2: val_accuracy improved from 0.47953 to 0.52087, saving model to /kaggle/working/v9/densenet201_v9_phase2b_best.keras

Epoch 2: finished saving model to /kaggle/working/v9/densenet201_v9_phase2b_best.keras
367/367 ━━━━━━━━━━━━━━━━━━━━ 80s 218ms/step - accuracy: 0.5305 - loss: 0.0458 - val_accuracy: 0.5209 - val_loss: 0.0260 - learning_rate: 5.0000e-05
Epoch 3/35
367/367 ━━━━━━━━━━━━━━━━━━━━ 0s 182ms/step - accuracy:

## 7. Curvas de entrenamiento

In [12]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Concatenar historial de fases
all_acc     = h1.history['accuracy']     + h2a.history['accuracy']     + h2b.history['accuracy']
all_val_acc = h1.history['val_accuracy'] + h2a.history['val_accuracy'] + h2b.history['val_accuracy']
all_loss    = h1.history['loss']         + h2a.history['loss']         + h2b.history['loss']
all_val_loss= h1.history['val_loss']     + h2a.history['val_loss']     + h2b.history['val_loss']

ep = range(1, len(all_acc) + 1)
p1_end  = len(h1.history['accuracy'])
p2a_end = p1_end + len(h2a.history['accuracy'])

for ax, train_vals, val_vals, ylabel in [
    (axes[0], all_acc,  all_val_acc,  'Accuracy'),
    (axes[1], all_loss, all_val_loss, 'Loss'),
]:
    ax.plot(ep, train_vals, label='Train', color='#1a6e35')
    ax.plot(ep, val_vals,   label='Val',   color='#c85a08')
    ax.axvline(p1_end,  color='gray', linestyle=':', alpha=0.7, label='Fase 2a')
    ax.axvline(p2a_end, color='navy', linestyle=':', alpha=0.7, label='Fase 2b')
    ax.set_xlabel('Época')
    ax.set_ylabel(ylabel)
    ax.set_title(f'{ylabel} por época')
    ax.legend()

plt.suptitle('v9 — FocalLossAlpha balanceado', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'training_curves_v9.png'), dpi=130, bbox_inches='tight')
plt.show()
print('Guardada: training_curves_v9.png')


Guardada: training_curves_v9.png


## 8. Búsqueda del umbral óptimo para mel

In [13]:
print('Calculando probabilidades en validation...')
val_probs = model.predict(X_val_clean, batch_size=64, verbose=1)

thresholds = np.arange(0.05, 0.60, 0.01)
f1_mel_scores   = []
f1_macro_scores = []

for t in thresholds:
    preds = np.argmax(val_probs, axis=1).copy()
    preds[val_probs[:, MEL_IDX] > t] = MEL_IDX
    f1_mel_scores.append(
        f1_score(y_val_int == MEL_IDX, preds == MEL_IDX, zero_division=0))
    f1_macro_scores.append(
        f1_score(y_val_int, preds, average='macro', zero_division=0))

best_t_macro = thresholds[np.argmax(f1_macro_scores)]
best_t_mel   = thresholds[np.argmax(f1_mel_scores)]

print(f'Umbral óptimo F1-macro: t={best_t_macro:.2f}  F1={max(f1_macro_scores):.4f}')
print(f'Umbral óptimo F1-mel:   t={best_t_mel:.2f}  F1-mel={max(f1_mel_scores):.4f}')

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(thresholds, f1_macro_scores, label='F1-macro', color='#1a6e35', linewidth=2)
ax.plot(thresholds, f1_mel_scores,   label='F1-mel',   color='#c85a08', linewidth=2)
ax.axvline(best_t_macro, color='#1a6e35', linestyle='--', alpha=0.7,
           label=f'Mejor macro t={best_t_macro:.2f}')
ax.axvline(best_t_mel, color='#c85a08', linestyle='--', alpha=0.7,
           label=f'Mejor mel t={best_t_mel:.2f}')
ax.set_xlabel('Umbral para mel')
ax.set_ylabel('F1 Score')
ax.set_title('Búsqueda de umbral óptimo (val set) — v9')
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'threshold_search_v9.png'), dpi=150)
plt.show()


Calculando probabilidades en validation...
39/39 ━━━━━━━━━━━━━━━━━━━━ 66s 997ms/step
Umbral óptimo F1-macro: t=0.50  F1=0.6887
Umbral óptimo F1-mel:   t=0.50  F1-mel=0.4474


## 9. Evaluación en test set

In [14]:
print('Calculando probabilidades en test...')
test_probs = model.predict(X_test_clean, batch_size=64, verbose=1)

# Estrategia B: argmax directo
y_pred_B = np.argmax(test_probs, axis=1)
acc_B    = accuracy_score(y_test_int, y_pred_B)
f1_B     = f1_score(y_test_int, y_pred_B, average='macro', zero_division=0)
f1_mel_B = f1_score((y_test_int==MEL_IDX), (y_pred_B==MEL_IDX), zero_division=0)

print('=== Estrategia B — argmax (FocalLossAlpha balanceado) ===')
print(classification_report(y_test_int, y_pred_B,
                             target_names=CLASS_NAMES, zero_division=0))
print(f'Accuracy={acc_B:.4f}  F1-macro={f1_B:.4f}  F1-mel={f1_mel_B:.4f}')

print()

# Estrategia A+B: argmax + umbral
THRESHOLD_MEL = best_t_macro
y_pred_AB = np.argmax(test_probs, axis=1).copy()
y_pred_AB[test_probs[:, MEL_IDX] > THRESHOLD_MEL] = MEL_IDX
acc_AB    = accuracy_score(y_test_int, y_pred_AB)
f1_AB     = f1_score(y_test_int, y_pred_AB, average='macro', zero_division=0)
f1_mel_AB = f1_score((y_test_int==MEL_IDX), (y_pred_AB==MEL_IDX), zero_division=0)

print(f'=== Estrategia A+B — umbral {THRESHOLD_MEL:.2f} ===')
print(classification_report(y_test_int, y_pred_AB,
                             target_names=CLASS_NAMES, zero_division=0))
print(f'Accuracy={acc_AB:.4f}  F1-macro={f1_AB:.4f}  F1-mel={f1_mel_AB:.4f}')


Calculando probabilidades en test...
21/21 ━━━━━━━━━━━━━━━━━━━━ 27s 1s/step
=== Estrategia B — argmax (FocalLossAlpha balanceado) ===
              precision    recall  f1-score   support

       akiec       0.90      0.62      0.73        42
         bcc       0.78      0.79      0.79        67
         bkl       0.46      0.78      0.58       142
          df       0.86      0.86      0.86        14
         mel       0.28      0.97      0.44       144
          nv       0.99      0.48      0.65       858
        vasc       0.78      1.00      0.88        18

    accuracy                           0.60      1285
   macro avg       0.72      0.79      0.70      1285
weighted avg       0.84      0.60      0.63      1285

Accuracy=0.6008  F1-macro=0.7021  F1-mel=0.4371

=== Estrategia A+B — umbral 0.50 ===
              precision    recall  f1-score   support

       akiec       0.90      0.62      0.73        42
         bcc       0.78      0.79      0.79        67
         bkl       0

## 10. Matrices de confusión

In [15]:
fig, axes = plt.subplots(1, 2, figsize=(18, 7))
for ax, y_pred, title in [
    (axes[0], y_pred_B,  f'B — FocalAlpha balanceado\nAcc={acc_B:.3f} F1={f1_B:.3f} F1-mel={f1_mel_B:.3f}'),
    (axes[1], y_pred_AB, f'A+B — FocalAlpha + Umbral {THRESHOLD_MEL:.2f}\nAcc={acc_AB:.3f} F1={f1_AB:.3f} F1-mel={f1_mel_AB:.3f}'),
]:
    cm = confusion_matrix(y_test_int, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='OrRd',
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
                linewidths=0.5, ax=ax)
    ax.set_title(title, fontsize=10)
    ax.set_ylabel('Clase real')
    ax.set_xlabel('Clase predicha')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'confusion_matrices_v9.png'), dpi=130, bbox_inches='tight')
plt.show()
print('Guardada: confusion_matrices_v9.png')


Guardada: confusion_matrices_v9.png


## 11. Resumen comparativo

In [16]:
print('=' * 65)
print('COMPARATIVA DE VERSIONES')
print('=' * 65)
print(f'{"Versión":25s} {"Accuracy":>10} {"F1-macro":>10} {"F1-mel":>10}')
print('-' * 60)
print(f'{"v5 (baseline)":25s} {0.6879:>10.4f} {0.4362:>10.4f} {0.0000:>10.4f}')
print(f'{"v7 (hair+oversample)":25s} {0.80:>10.4f} {"N/A":>10} {0.0000:>10.4f}')
print(f'{"v8-B (alpha extremo)":25s} {0.6864:>10.4f} {0.1933:>10.4f} {0.4797:>10.4f}')
print(f'{"v9-B (alpha balanceado)":25s} {acc_B:>10.4f} {f1_B:>10.4f} {f1_mel_B:>10.4f}')
print(f'{"v9-AB (+ umbral)":25s} {acc_AB:>10.4f} {f1_AB:>10.4f} {f1_mel_AB:>10.4f}')
print('=' * 65)

# Meta: superar v5 en F1-macro y tener mel F1 > 0
meta_f1_macro = 0.4362  # v5
best_f1 = max(f1_B, f1_AB)
best_mel = max(f1_mel_B, f1_mel_AB)
print(f'\nMeta F1-macro > {meta_f1_macro}: {"✓ SUPERADA" if best_f1 > meta_f1_macro else "✗ No superada"} ({best_f1:.4f})')
print(f'Meta F1-mel > 0:              {"✓ SUPERADA" if best_mel > 0 else "✗ No superada"} ({best_mel:.4f})')

print('\nArchivos generados:')
for f in sorted(os.listdir(OUTPUT_DIR)):
    fp = os.path.join(OUTPUT_DIR, f)
    print(f'  {f:50s} {os.path.getsize(fp)/1e6:.1f} MB')


COMPARATIVA DE VERSIONES
Versión                     Accuracy   F1-macro     F1-mel
------------------------------------------------------------
v5 (baseline)                 0.6879     0.4362     0.0000
v7 (hair+oversample)          0.8000        N/A     0.0000
v8-B (alpha extremo)          0.6864     0.1933     0.4797
v9-B (alpha balanceado)       0.6008     0.7021     0.4371
v9-AB (+ umbral)              0.6008     0.7021     0.4371

Meta F1-macro > 0.4362: ✓ SUPERADA (0.7021)
Meta F1-mel > 0:              ✓ SUPERADA (0.4371)

Archivos generados:
  confusion_matrices_v9.png                          0.1 MB
  densenet201_v9_phase1_best.keras                   81.7 MB
  densenet201_v9_phase2a_best.keras                  88.7 MB
  densenet201_v9_phase2b_best.keras                  110.4 MB
  threshold_search_v9.png                            0.1 MB
  training_curves_v9.png                             0.1 MB
